In [ ]:
# ==========================================
# CELL 1 — Bootstrap
# ------------------------------------------
# • Clones your song-generator repo from GitHub
# • Registers the notebook/ folder on sys.path
# • Runs stage 1 (GPU check, system deps, secrets, audio.cpp binary)
# ==========================================
import os
import sys
import shutil
import subprocess

# --- Resolve repo URL from Kaggle Secrets (or use the fallback below) ---
try:
    from kaggle_secrets import UserSecretsClient
    GITHUB_REPO = UserSecretsClient().get_secret("GITHUB_REPO").strip()
except Exception:
    GITHUB_REPO = "https://github.com/YOUR_USERNAME/song-generator.git"  # ← edit me

# --- Fresh clone every session (deletes any previous clone) ---
REPO_DIR = "/kaggle/working/_repo"
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

print(f"Cloning {GITHUB_REPO} → {REPO_DIR}")
res = subprocess.run(
    ["git", "clone", "--depth", "1", GITHUB_REPO, REPO_DIR],
    capture_output=True, text=True,
)
if res.returncode != 0:
    raise RuntimeError(f"git clone failed:\n{res.stderr}")

# --- Register notebook/ on sys.path so `from stages import ...` works ---
sys.path.insert(0, os.path.join(REPO_DIR, "notebook"))

# --- Run stage 1 ---
from stages import s01_setup
s01_setup.run()

In [ ]:
# ==========================================
# CELL 2 — Model Download & Configuration
# ------------------------------------------
# • Downloads YuE2 GGUF weights from Hugging Face (or uses pre-mounted ones)
# • Writes server_gpu0.json and server_gpu1.json into /kaggle/working/audiocpp
# ==========================================
from stages import s02_model
s02_model.run()

In [ ]:
# ==========================================
# CELL 3 — Start Servers
# ------------------------------------------
# • Detects GPU count and chooses single/dual-GPU mode
# • Launches ./audiocpp_server on port 8080 (and 8081 if dual)
# • Waits for /health to return 200 on each
# ==========================================
from stages import s03_servers
s03_servers.run()

In [ ]:
# ==========================================
# CELL 4 — Ingest Job Queue
# ------------------------------------------
# • Downloads queue/jobs.json from the HF dataset vault
# • Saves it to /kaggle/working/queue_in/jobs.json
# • Prints a Rich table showing all jobs + pending count
# • Populates state.pending_jobs for Cell 5
# ==========================================
from stages import s04_queue
s04_queue.run()

In [ ]:
# ==========================================
# CELL 5 — Audio Generation (YuE2)
# ------------------------------------------
# • Spawns one worker thread per available GPU
# • Each worker pulls pending jobs and POSTs to /v1/tasks/run
# • Saves {slug}.wav and {slug}-score.abc into /kaggle/working/output/
# • Updates job status (audio_generated / failed) in jobs.json
# ==========================================
from stages import s05_generate
s05_generate.run()

In [ ]:
# ==========================================
# CELL 6 — Video Studio, Bundle & Sync
# ------------------------------------------
# • For each WAV: creates a Title-Case PNG thumbnail + 1080p MP4
# • Bundles all outputs + a manifest.json into batch-<timestamp>.zip
# • Uploads ZIP (to outputs/) AND updated ledger (queue/jobs.json)
#   to Hugging Face in ONE atomic commit
# ==========================================
from stages import s06_bundle
s06_bundle.run()

In [ ]:
# ==========================================
# CELL 7 — Verification & Shutdown
# ------------------------------------------
# • Lists all ZIP bundles waiting in the HF vault
# • Kills all audiocpp_server processes
# • Tells you to run `python main.py collect` locally to pull the bundles
# ==========================================
from stages import s07_verify
s07_verify.run()